**ENCODER ARCHITECTURE - phase 1 (training the model to learn what a normal thermal foot looks like )**

so basically we save an encoder here which converts a raw thermal image into meaningful maps these maps are useful in ans where is foot where is hot or cold 
it stores only no not images 

it has learned :
edge like thermal changes 
smooth heat transitions
symmetrry between left and right foot 

so encoder saves the information  required to interpret thermal images 


now lets deeply understand the model :


the dataset class:

we know each csv is a thermal map so t(x,y) = pixel temp at (x,y)
we convert the dataframe to numpy arrays 
then we clip the temp
then we normalise the clipped thermal_map

now we resize the images and use cubic interpolation:
interpolation is basically when we resize we create new pixels and to predict the value for this pixels we calculate the weighted average of nearby pixels, using inter cubic we increase the area which we will consider for average calculation 

then we add a channel dimesion which is one for thermal and 3 for rgb we want thermal one



appply_ramdom_mask:

we will deliberately hide some patches in image and force the model to understand the thermal structure of foot 

the function takes batch of thermal images
B = batch size
C=1 (thermal channel)
H,W = spatial dimensions (e.g., 256 × 256)


so 50-60% is missing and missing regions are contiguou, masking is random every batch 
the encoder learn :
thermal continuity 
spatial structure(the consistent anatomical shape of foot)
contextual inference :
given heel + toes infer midfoot temperature 
this is representation learning where model itself extracts the features from the raw data to perform the given task 


model part:

we are defining a function which takes input incomplete thermal image and output reconstructed thermal image
now unet has two things
1. encoder = representation learning
2. decoder = image synthesis

we used resnet 18 as encoder, and we didnt use the imagenet weights as we wanted model to learn the thermal features only not edges or colour blobs in rgb images
mse enforces numerical thermal accuracy

as a result the encoder learns thermal physiological representation not edges not textures not noise 

so in future the asymmetry for the model means true physiological deviation not artifacts 


the loop:

we take a batch of image and apply mask then pass it to model, the model only sees the zeros it does not know where the mask is 
now since the input is masked the decoder reconstruct missing areas 

so encoder must encode :
1. foot geometry 
2. themal smoothness
3. toes + heel + midfoot temp

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split, Dataset
from torch.cuda.amp import GradScaler, autocast  
import segmentation_models_pytorch as smp
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import cv2

# --- 1. CONFIGURATION ---
BATCH_SIZE = 16          # the model sees 16 thermal images at the same time 
LEARNING_RATE = 1e-4 # controls how fast model changes it knowledge 
EPOCHS = 80
VAL_SPLIT = 0.2
NUM_WORKERS = 0          # Set this to the number of CPU cores you have (usually 4 or 8)
PATCH_SIZE = 16  #16x16 size patches 
MASK_RATIO = 0.6 # it means 60% of the foot is missing 

# Check for CUDA
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    print(f"✅ GPU Detected: {torch.cuda.get_device_name(0)}")
    torch.backends.cudnn.benchmark = True  # Optimizes specific convolution algos
else:
    DEVICE = torch.device("cpu")
    

# this class ensures the datset is loaded correctly and is consistent as we are using both control and dm group so the model may learn the difference betweeen them but our goal is to make the model learn a simple thermal image of a foot so we do clipping also here 
class INAOEThermalDataset(Dataset):
    def __init__(self, root_dir, img_size=(256, 256)):
        self.root_dir = root_dir
        self.img_size = img_size
        self.file_list = self._load_files()

    def _load_files(self): # finds all the csv files 
        csv_files = []
        for root, dirs, files in os.walk(self.root_dir):
            for file in files:
                if file.endswith(".csv"):
                    if any(x in file for x in ['LCA', 'LPA', 'MCA', 'MPA']): # we dont include them as they are plantar regions 
                        continue
                    csv_files.append(os.path.join(root, file))
        return csv_files

    def __len__(self):
        return len(self.file_list)

    def __getitem__(self, idx):
        csv_path = self.file_list[idx] #every csv is a thermal map
        try:
            # Load and process
            df = pd.read_csv(csv_path, header=None)
            thermal_matrix = df.values.astype(np.float32) #convert dataframe to numpy arrays 
            
            # below 20 is air and above 40 is noise 
            thermal_matrix = np.clip(thermal_matrix, 20.0, 40.0)
            normalized_img = (thermal_matrix - 20.0) / (40.0 - 20.0) 
            
            # Resize image as allimage should be of same size 
            resized_img = cv2.resize(normalized_img, (self.img_size[1], self.img_size[0]), interpolation=cv2.INTER_CUBIC)
            
            # Add channel dimension
            img_tensor = torch.from_numpy(resized_img).unsqueeze(0) 
            return img_tensor
            
        except Exception as e:
            print(f"Error loading {csv_path}: {e}")
            return torch.zeros((1, self.img_size[0], self.img_size[1]))


def apply_random_mask(images, patch_size=32, mask_ratio=0.5):
    
    B, C, H, W = images.shape
    
    mask = torch.ones((B, 1, H, W), device=images.device)
    
    h_patches = H // patch_size 
    w_patches = W // patch_size
    num_patches = h_patches * w_patches
    num_masked = int(mask_ratio * num_patches) #decides how many patches to hide 

    
    for b in range(B):
        mask_indices = np.random.choice(num_patches, num_masked, replace=False) #masking random patches
        for idx in mask_indices:
            row = (idx // w_patches) * patch_size
            col = (idx % w_patches) * patch_size
            mask[b, :, row:row+patch_size, col:col+patch_size] = 0.0 #random patches
    #apply mask        
    masked_images = images * mask
    loss_mask = 1 - mask
    return masked_images, loss_mask


def visualize_results(masked, reconstructed, original, epoch, mode='train'):
    # Detach and move to CPU for plotting
    img_m = masked[0, 0].cpu().detach().numpy()
    img_r = reconstructed[0, 0].cpu().detach().numpy()
    img_o = original[0, 0].cpu().detach().numpy()

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    axes[0].imshow(img_m, cmap='inferno'); axes[0].set_title("Input (Masked)")
    axes[1].imshow(img_r, cmap='inferno'); axes[1].set_title("Reconstruction")
    axes[2].imshow(img_o, cmap='inferno'); axes[2].set_title("Ground Truth")
    
    plt.suptitle(f"{mode.upper()} - Epoch {epoch}")
    os.makedirs('results', exist_ok=True)
    plt.savefig(f"results/{mode}_epoch_{epoch}.png")
    plt.close()


if __name__ == '__main__': 
    full_dataset = INAOEThermalDataset(root_dir='./INAOE_Dataset', img_size=(256, 256))

    val_size = int(len(full_dataset) * VAL_SPLIT)
    train_size = len(full_dataset) - val_size
    train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])


    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, 
                              num_workers=NUM_WORKERS, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, 
                            num_workers=NUM_WORKERS, pin_memory=True)

    print(f"Data Loaded: {train_size} Train | {val_size} Val")

    model = smp.Unet( #unet helps with pixel level prediction, strong spatial reconstruction
        encoder_name="resnet18",     # encoder basically generlaises the image by combining group of small regions of pixel into a single region by doing this the quality reduces but the information remains 
        encoder_weights=None,       # we have not used imagenet here coz they are rgb images     
        in_channels=1,        # we have only one input channel and that is temperature not any colors           
        classes=1,                      
        activation='sigmoid'
    ).to(DEVICE)

    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
    criterion = nn.MSELoss(reduction='none') 
    

    scaler = GradScaler()


    best_val_loss = float('inf')
    print(f"Starting Training on {DEVICE} with Mixed Precision...")

    for epoch in range(EPOCHS):
        model.train()
        train_loss = 0
        
        for batch_idx, original_images in enumerate(train_loader):
            original_images = original_images.to(DEVICE, non_blocking=True)
            
            # Smaller Patch Size
            masked_images, loss_mask = apply_random_mask(original_images, patch_size=PATCH_SIZE, mask_ratio=MASK_RATIO)
            
            with autocast():
                reconstructed_images = model(masked_images)
                
                # CHANGED: Global MSE Loss (Calculate on WHOLE image, not just mask)
                # This forces the model to blend the edges of the mask smoothly
                loss = criterion(reconstructed_images, original_images) 
                
                # We average the loss over all pixels now
                final_loss = loss.mean()

            optimizer.zero_grad()
            scaler.scale(final_loss).backward()
            scaler.step(optimizer)
            scaler.update()
            
            train_loss += final_loss.item()
        
        # --- VALIDATION (Keep same logic) ---
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for batch_idx, original_images in enumerate(val_loader):
                original_images = original_images.to(DEVICE, non_blocking=True)
                masked_images, _ = apply_random_mask(original_images, patch_size=PATCH_SIZE, mask_ratio=MASK_RATIO)
                
                with autocast():
                    reconstructed_images = model(masked_images)
                    loss = criterion(reconstructed_images, original_images)
                    val_loss += loss.mean().item()

                # Visualize first batch of every 10th epoch
                if batch_idx == 0 and (epoch + 1) % 10 == 0:
                    visualize_results(masked_images, reconstructed_images, original_images, epoch+1, mode='val')

        avg_val_loss = val_loss / len(val_loader)
        print(f"Epoch [{epoch+1}/{EPOCHS}] Loss: {avg_val_loss:.6f}")
        
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(model.encoder.state_dict(), "best_inaoe_encoder_v2.pth")

    print("Improved Pretraining Complete.")

**DATASET PREPERATION**

In [ ]:
import json
import cv2
import numpy as np
import os
import glob
from tqdm import tqdm

# --- CONFIGURATION ---
# Update these paths to match your actual folders
ROOT_DIR = "./STANDUP_Database" 
OUTPUT_DIR = "./STANDUP_Ready_Phase2"  # Final destination for training data
IMG_SIZE = (256, 256)

def get_corresponding_thermal_path(rgb_json_path):
    """
    For Healthy Group: Finds the thermal image that matches the RGB JSON.
    Assumes structure: 
       .../healthy/RGB/Subject01.json
       .../healthy/thermal/Subject01.png (or .jpg)
    """
    rgb_folder = os.path.dirname(rgb_json_path)
    parent_folder = os.path.dirname(rgb_folder)
    thermal_folder = os.path.join(parent_folder, "thermal")
    
    filename_no_ext = os.path.splitext(os.path.basename(rgb_json_path))[0]
    
    possible_exts = ['.png', '.jpg', '.jpeg', '.bmp', '.tif']
    
    for ext in possible_exts:
        test_path = os.path.join(thermal_folder, filename_no_ext + ext)
        if os.path.exists(test_path):
            return test_path
            
    return None

def process_and_save(image_path, json_path, save_folder_name):
    # 1. Load Image (Thermal)
    img = cv2.imread(image_path)
    if img is None:
        print(f"❌ Error: Could not load image {image_path}")
        return

    # 2. Load Mask JSON
    try:
        with open(json_path, "r") as f:
            data = json.load(f)
    except Exception as e:
        print(f"❌ Error loading JSON {json_path}: {e}")
        return

    # 3. Create Mask from Polygons
    h, w = img.shape[:2]
    mask = np.zeros((h, w), dtype=np.uint8)
    
    for shape in data['shapes']:
        points = shape['points']
        pts = np.array(points, dtype=np.int32).reshape((-1, 1, 2))
        cv2.fillPoly(mask, [pts], 255)

    # 4. Apply Mask (Blackout Background)
    masked_img = cv2.bitwise_and(img, img, mask=mask)

    # 5. Extract Feet (Bounding Box Logic)
    # Find contours on the mask we just created
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    # Sort by area (largest first) to find the two feet
    contours = sorted(contours, key=cv2.contourArea, reverse=True)[:2]
    
    feet_crops = []
    centers_x = []

    for cnt in contours:
        x, y, w, h = cv2.boundingRect(cnt)
        
        # Add padding (optional, ensures we don't crop toes too tight)
        pad = 10
        x = max(0, x - pad); y = max(0, y - pad)
        w = min(img.shape[1] - x, w + 2*pad)
        h = min(img.shape[0] - y, h + 2*pad)
        
        crop = masked_img[y:y+h, x:x+w]
        
        # Resize to standard size for AI
        crop_resized = cv2.resize(crop, IMG_SIZE, interpolation=cv2.INTER_CUBIC)
        
        feet_crops.append(crop_resized)
        centers_x.append(x + w//2)

    # 6. Save Left and Right
    if len(feet_crops) == 2:
        # Determine L/R based on x-coordinate
        # Image-Left (Smaller X) is usually Patient's Right Foot
        # Image-Right (Larger X) is usually Patient's Left Foot
        if centers_x[0] < centers_x[1]:
            # crop[0] is on the left side of the image
            foot_left_side_of_img = feet_crops[0]
            foot_right_side_of_img = feet_crops[1]
        else:
            foot_left_side_of_img = feet_crops[1]
            foot_right_side_of_img = feet_crops[0]

        # Construct Output Path
        # We put them in specific folders for the Loader (healthy/diabetic)
        final_out_dir = os.path.join(OUTPUT_DIR, save_folder_name)
        os.makedirs(final_out_dir, exist_ok=True)
        
        base_name = os.path.splitext(os.path.basename(image_path))[0]
        
        # Save L and R
        # Note: We name them by visual position for the Siamese Loader
        cv2.imwrite(os.path.join(final_out_dir, f"{base_name}_L.png"), foot_left_side_of_img)
        cv2.imwrite(os.path.join(final_out_dir, f"{base_name}_R.png"), foot_right_side_of_img)

# --- MAIN EXECUTION LOOP ---
print("🚀 Starting Processing...")

# 1. PROCESS HEALTHY (RGB Masks -> Thermal Images)
healthy_rgb_path = os.path.join(ROOT_DIR, "healthy", "RGB")
# Find all JSONs in RGB folder
healthy_jsons = glob.glob(os.path.join(healthy_rgb_path, "*.json"))

print(f"Found {len(healthy_jsons)} Healthy masks in RGB folder.")

for json_file in tqdm(healthy_jsons, desc="Processing Healthy"):
    # Find the matching THERMAL image
    thermal_img_path = get_corresponding_thermal_path(json_file)
    
    if thermal_img_path:
        process_and_save(thermal_img_path, json_file, "healthy")
    else:
        print(f"⚠️ Warning: No thermal match found for {os.path.basename(json_file)}")

# 2. PROCESS DIABETIC (Thermal Masks -> Thermal Images)
# We look in R0, R1, R2 folders inside 'diabetic'
diabetic_root = os.path.join(ROOT_DIR, "diabetic")

for r_group in ['R0', 'R1', 'R2']:
    # Path to thermal folder (where JSONs are now)
    group_path = os.path.join(diabetic_root, r_group, "thermal")
    
    if os.path.exists(group_path):
        # Find JSONs
        diabetic_jsons = glob.glob(os.path.join(group_path, "*.json"))
        print(f"Found {len(diabetic_jsons)} masks in Diabetic {r_group}.")
        
        for json_file in tqdm(diabetic_jsons, desc=f"Processing {r_group}"):
            # For diabetic, image is in the SAME folder as JSON
            # We just need to find the matching image file
            base_name = os.path.splitext(json_file)[0]
            image_path = None
            for ext in ['.png', '.jpg', '.jpeg']:
                if os.path.exists(base_name + ext):
                    image_path = base_name + ext
                    break
            
            if image_path:
                # We save R0 and R2 to specific folders for our experiments
                # R0 -> diabetic_R0 (For testing)
                # R2 -> diabetic_R2 (For training)
                # R1 -> diabetic_R1 (If you decided to keep it)
                process_and_save(image_path, json_file, f"diabetic_{r_group}")

print("\n✅ All processing complete. Check 'STANDUP_Ready_Phase2' folder.")

**R2, R0 run - PHASE 2**

first we make the pairs of extracted foot left and right then we label them o and 1
healthy will get 0 
diabetic will get 1 

the siamese network architecture :

STN:
it is an aligner, it looks at the right foot and physically rotates it to overlay perfectly in the left foot 

encoder :
it converts the raw images into feature vectors fl and fr, we inject the phase 1 weights here and then the model starts with the full knowledge of a themal foot, it already knows the top curve part is toes and the bright spot is heat
it outputs a feauture vector a list of numbers that summarizes the thermal pattern of the foot the siamese network then compares these two numbers 


comparator :
it calculates the absolute difference between the left foot and the right foot and output a sigmoid score in which high difference =  high risk of dfu 

what we are targetting is the dark grey paet of the foot not light gray not white 


now for the training part we trained two models one with r0 and another with r2 as the main label 
the r2 model gave almost the same results as the r0 but with the less amount of data and is more confident with the predictions this proves the r2 supermacy over the r1 



lets dive deep now 

after the phase 1 unsupervised learning we will move to phase 2 assymetry difference learning which will be supervised 

siamese dataset class:
in phase 2 we play with the images by changing brightness and rotation, the phase one encoder learns the global thermal structure not pixel exactness so it remains robust, if encoder has memorized exact pixel statistics the brightness change will destroy the performance 

if we dont alter with brightness the model might learn high global average temp so its diabetic
the aug forces model to learn the asymmetry patterns 

then we make pairs of the foot 


stn class:

we are calculating the feature difference of left and right foot but if left heel pixel aligns with right toe pixel feature difference will be of no use so our task here is to align xl and xr

stn here does not try to match images visually instead it tries to find the geometric transform that improves fl-fr, it learns alignment in feature space not pixel space 
pixel wise alignment forces the stn to align every pixel perfectly
but the feature difference will align the heel to heel even if lightning changes as its focsuing on reducing the overall feature differnece not the pixel difference 

imagine two pixels with same temp slightly misaligned by 2 pixels, the pixel diff will give artificial diff
but if we do feature mapping with encoder, each feature value represents the local temp average so the diff wont be two large 



siamese class :

we load the phase 1 weights here in the encoder so the encoder already knows thermal continuity , foot structure , this reduces the load on phase 2
1. we flip the right foot
2. align it with stn 
3. extract feature map using the encoder 
4. calculate the global feature difference 

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp
import cv2
import os
import numpy as np
import matplotlib.pyplot as plt
import albumentations as A #data augmentations 
from albumentations.pytorch import ToTensorV2
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix


EXPERIMENT_MODE = "R2"  #change it to R0 for testing on R0

BATCH_SIZE = 8 #at a time the model sees 8 images 
EPOCHS = 30
LR = 1e-4
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATA_DIR = "./STANDUP_Ready_Phase2"
PRETRAINED_PATH = "best_inaoe_encoder_v2.pth"
SAVE_NAME = f"siamese_{EXPERIMENT_MODE.lower()}_best_metrics.pth" #saves the best model based on f1 

print(f"🚀 STARTING EXPERIMENT: {EXPERIMENT_MODE}")
print(f"📊 Metrics will include: Accuracy, Precision, Recall, F1, AUROC")

#pairing class
class SiameseFootDataset(Dataset):
    def __init__(self, root_dir, mode='train', experiment_mode='R2'):
        self.root_dir = root_dir
        self.mode = mode
        self.experiment_mode = experiment_mode
        self.pairs = self._make_pairs()
        
        
        if self.mode == 'train':
            self.transform = A.Compose([
                
                # applies amall rotation
                A.Rotate(limit=10, p=0.5), 
                
                # Shift/Scale is fine, STN handles this
                A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.05, rotate_limit=0, p=0.5),
                
                # Thermal intensity jitter
                A.RandomBrightnessContrast(brightness_limit=0.1, contrast_limit=0.1, p=0.5),
                
                # resizing and normalization
                A.Resize(256, 256),
                A.Normalize(mean=(0.5,), std=(0.5,)),
                ToTensorV2()
            ])
        else: # for validation we resize and normalize only 
            self.transform = A.Compose([
                A.Resize(256, 256),
                A.Normalize(mean=(0.5,), std=(0.5,)),
                ToTensorV2()
            ])
        
    def _make_pairs(self):
        pairs = []
        for root, dirs, files in os.walk(self.root_dir):
            for f in files:
                if "_L.png" in f: #finds the left foot
                    l_path = os.path.join(root, f)
                    r_path = os.path.join(root, f.replace("_L.png", "_R.png")) #matches with right foot 
                    
                    if os.path.exists(r_path):
                        folder_name = os.path.basename(root).lower()
                        label = None
                        
                        if "healthy" in folder_name:
                            label = 0.0
                        elif self.experiment_mode == "R2" and "diabetic_r2" in folder_name:
                            label = 1.0
                        elif self.experiment_mode == "R0" and "diabetic_r0" in folder_name:
                            label = 1.0
                            
                        if label is not None:
                            pairs.append((l_path, r_path, label))
        return pairs

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        l_path, r_path, label = self.pairs[idx]
        img_l = cv2.imread(l_path, cv2.IMREAD_GRAYSCALE)
        img_r = cv2.imread(r_path, cv2.IMREAD_GRAYSCALE)
        
        if img_l is None or img_r is None:
            return torch.zeros(1, 256, 256), torch.zeros(1, 256, 256), torch.tensor(label).float()

        transformed_l = self.transform(image=img_l)["image"]
        transformed_r = self.transform(image=img_r)["image"]
        return transformed_l, transformed_r, torch.tensor(label).float()

# small cnn 
class SpatialTransformer(nn.Module):
    def __init__(self):
        super(SpatialTransformer, self).__init__()
        self.localization = nn.Sequential(
            nn.Conv2d(1, 8, kernel_size=7), nn.MaxPool2d(2, stride=2), nn.ReLU(True),
            nn.Conv2d(8, 10, kernel_size=5), nn.MaxPool2d(2, stride=2), nn.ReLU(True)
        )
        self.fc_loc = nn.Sequential(
            nn.Linear(10 * 60 * 60, 32), nn.ReLU(True), nn.Linear(32, 3 * 2)
        )
        self.fc_loc[2].weight.data.zero_()
        self.fc_loc[2].bias.data.copy_(torch.tensor([1, 0, 0, 0, 1, 0], dtype=torch.float))

    def forward(self, x):
        xs = self.localization(x)
        xs = xs.view(-1, 10 * 60 * 60)
        theta = self.fc_loc(xs)
        theta = theta.view(-1, 2, 3)
        grid = F.affine_grid(theta, x.size(), align_corners=True)
        x_sampled = F.grid_sample(x, grid, align_corners=True)
        return x_sampled

class SiameseNetwork(nn.Module):
    def __init__(self, pretrained_path=None):
        super(SiameseNetwork, self).__init__()
        self.stn = SpatialTransformer()
        self.encoder = smp.Unet(encoder_name="resnet18", in_channels=1, classes=1).encoder
        
        if pretrained_path and os.path.exists(pretrained_path):
            print(f"✅ Loading Phase 1 Weights from {pretrained_path}")
            self.encoder.load_state_dict(torch.load(pretrained_path))
        else:
            print("⚠️ Warning: Training from scratch (No Phase 1 weights found).")
            
        self.classifier = nn.Sequential(
            nn.Linear(512, 128), nn.ReLU(),
            nn.Linear(128, 1), nn.Sigmoid()
        )

    def forward(self, l, r):
        r_flipped = torch.flip(r, dims=[3])
        r_aligned = self.stn(r_flipped)
        f_l = self.encoder(l)[-1].mean(dim=(2,3)) 
        f_r = self.encoder(r_aligned)[-1].mean(dim=(2,3))
        diff = torch.abs(f_l - f_r)
        score = self.classifier(diff)
        return score

# --- 3. HELPER: METRIC CALCULATION ---
def calculate_metrics(loader, model, device):
    model.eval()
    all_labels = []
    all_scores = []
    
    with torch.no_grad():
        for l, r, label in loader:
            l, r = l.to(device), r.to(device)
            score = model(l, r).cpu().numpy().flatten()
            label = label.numpy().flatten()
            
            all_scores.extend(score)
            all_labels.extend(label)
            
    all_scores = np.array(all_scores)
    all_labels = np.array(all_labels)
    
    # Binarize predictions (Threshold 0.5)
    preds = (all_scores > 0.5).astype(int)
    
    # Calculate Metrics
    acc = accuracy_score(all_labels, preds)
    prec = precision_score(all_labels, preds, zero_division=0)
    rec = recall_score(all_labels, preds, zero_division=0)
    f1 = f1_score(all_labels, preds, zero_division=0)
    
    # AUROC requires both classes. If val set is tiny and missing a class, return 0.5
    try:
        auc = roc_auc_score(all_labels, all_scores)
    except:
        auc = 0.5 

    return acc, prec, rec, f1, auc, all_labels, preds

# --- 4. MAIN EXECUTION ---
if __name__ == "__main__":
    full_dataset = SiameseFootDataset(DATA_DIR, mode='train', experiment_mode=EXPERIMENT_MODE)
    if len(full_dataset) == 0:
        print("❌ No images found!")
        exit()

    train_size = int(0.8 * len(full_dataset))
    val_size = len(full_dataset) - train_size
    train_dataset, val_dataset = torch.utils.data.random_split(full_dataset, [train_size, val_size])
    
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
    
    model = SiameseNetwork(pretrained_path=PRETRAINED_PATH).to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=LR)
    criterion = nn.BCELoss()
    
    best_f1 = 0.0 # We will save based on F1 Score now (better than Accuracy for medical)

    print(f"🚀 Training {EXPERIMENT_MODE} Model...")
    
    for epoch in range(EPOCHS):
        model.train()
        train_loss = 0
        
        for l, r, label in train_loader:
            l, r, label = l.to(DEVICE), r.to(DEVICE), label.to(DEVICE).unsqueeze(1)
            optimizer.zero_grad()
            score = model(l, r)
            loss = criterion(score, label)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
            
        # --- VALIDATION EPOCH ---
        # We calculate full metrics every epoch to track progress
        acc, prec, rec, f1, auc, _, _ = calculate_metrics(val_loader, model, DEVICE)
        
        avg_loss = train_loss / len(train_loader)
        
        print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {avg_loss:.4f} | Acc: {acc:.2f} | F1: {f1:.2f} | AUROC: {auc:.2f}")
        
        # Save Best Model based on F1 Score (Balance of Precision/Recall)
        if f1 >= best_f1:
            best_f1 = f1
            torch.save(model.state_dict(), SAVE_NAME)
            print(f"    💾 New Best F1: {f1:.4f} (Saved)")

    # --- FINAL EVALUATION ON BEST MODEL ---
    print("\n🏆 FINAL EVALUATION (Best Model):")
    model.load_state_dict(torch.load(SAVE_NAME)) # Load the winner
    acc, prec, rec, f1, auc, y_true, y_pred = calculate_metrics(val_loader, model, DEVICE)
    
    print(f"--------------------------------")
    print(f"Experiment: {EXPERIMENT_MODE}")
    print(f"Accuracy:   {acc:.4f}")
    print(f"Precision:  {prec:.4f}")
    print(f"Recall:     {rec:.4f}")
    print(f"F1 Score:   {f1:.4f}")
    print(f"AUROC:      {auc:.4f}")
    print(f"--------------------------------")
    print("Confusion Matrix:")
    print(confusion_matrix(y_true, y_pred))

**foot heatmaps**

now to map the abnormal regions we need to extract the spatial difference map from the model before it crushes everything into a single score 

in classification task we computed fl-fr, now here instead of global asymmetry thermal measure we compute spatial assymetry field 

we will extract feature maps and we will calculate absolute difference per spatial location

so heatmap generated is local feature space distance field not pixel temp difference

if we do pixel difference it will show alignment noise pixel jitter and brightness noise

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import segmentation_models_pytorch as smp
import cv2
import os
import numpy as np
import matplotlib.pyplot as plt
import albumentations as A
from albumentations.pytorch import ToTensorV2

# --- 1. GLOBAL CONFIGURATION ---
TARGET_ID = "R2_0053"   # Which patient?
TARGET_SIDE = "R"       # Which foot to visualize? ("L" or "R")

MODEL_PATH = "siamese_r2_best_metrics.pth"
BASE_DIR = "./STANDUP_Ready_Phase2"
THERMAL_DIR = os.path.join(BASE_DIR, "diabetic_R2")
RGB_DIR = os.path.join(BASE_DIR, "RGB_Matched") # The folder we just created

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- 2. MODEL DEFINITION (Must match saved weights exactly) ---
class SpatialTransformer(nn.Module):
    def __init__(self):
        super(SpatialTransformer, self).__init__()
        self.localization = nn.Sequential(
            nn.Conv2d(1, 8, kernel_size=7), nn.MaxPool2d(2, stride=2), nn.ReLU(True),
            nn.Conv2d(8, 10, kernel_size=5), nn.MaxPool2d(2, stride=2), nn.ReLU(True)
        )
        self.fc_loc = nn.Sequential(
            nn.Linear(10 * 60 * 60, 32), nn.ReLU(True), nn.Linear(32, 3 * 2)
        )
        self.fc_loc[2].weight.data.zero_()
        self.fc_loc[2].bias.data.copy_(torch.tensor([1, 0, 0, 0, 1, 0], dtype=torch.float))

    def forward(self, x):
        xs = self.localization(x)
        xs = xs.view(-1, 10 * 60 * 60)
        theta = self.fc_loc(xs)
        theta = theta.view(-1, 2, 3)
        grid = F.affine_grid(theta, x.size(), align_corners=True)
        x_sampled = F.grid_sample(x, grid, align_corners=True)
        return x_sampled

class SiameseNetwork(nn.Module):
    def __init__(self):
        super(SiameseNetwork, self).__init__()
        self.stn = SpatialTransformer()
        self.encoder = smp.Unet(encoder_name="resnet18", in_channels=1, classes=1).encoder
        self.classifier = nn.Sequential(
            nn.Linear(512, 128), nn.ReLU(),
            nn.Linear(128, 1), nn.Sigmoid()
        )

    # --- NEW FUNCTION: Extract the Heatmap instead of the Score ---
    def get_heatmap(self, l, r):
        # 1. Flip & Align Right Foot (Standard procedure)
        r_flipped = torch.flip(r, dims=[3])
        r_aligned = self.stn(r_flipped)
        
        # 2. Extract Spatial Features (Do NOT use .mean() yet)
        # We get the 512x8x8 feature maps from the last layer of ResNet18
        f_l_map = self.encoder(l)[-1]  
        f_r_map = self.encoder(r_aligned)[-1]
        
        # 3. Calculate Absolute Difference at every pixel location
        diff_map = torch.abs(f_l_map - f_r_map)
        
        # 4. Collapse 512 channels into 1 Heatmap (Average intensity)
        heatmap = torch.mean(diff_map, dim=1, keepdim=True)
        
        # 5. Upscale Heatmap back to 256x256
        heatmap = F.interpolate(heatmap, size=(256, 256), mode='bilinear', align_corners=False)
        
        return heatmap

# --- 3. EXECUTION ---
def visualize_ulcer_zones():
    print(f"🔍 Analyzing Patient: {TARGET_ID} | Side: {TARGET_SIDE}")

    # A. Load Model
    model = SiameseNetwork().to(DEVICE)
    if os.path.exists(MODEL_PATH):
        model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
        print("✅ Model loaded successfully.")
    else:
        print("❌ Model weights not found.")
        return
    model.eval()

    # B. Load Thermal Images (We need BOTH Left and Right to calculate asymmetry)
    path_l = os.path.join(THERMAL_DIR, f"{TARGET_ID}_L.png")
    path_r = os.path.join(THERMAL_DIR, f"{TARGET_ID}_R.png")
    
    if not os.path.exists(path_l) or not os.path.exists(path_r):
        print(f"❌ Missing thermal files for {TARGET_ID}")
        return

    # Preprocess Thermal (Standardize)
    transform = A.Compose([
        A.Resize(256, 256),
        A.Normalize(mean=(0.5,), std=(0.5,)),
        ToTensorV2()
    ])

    img_l_raw = cv2.imread(path_l, cv2.IMREAD_GRAYSCALE)
    img_r_raw = cv2.imread(path_r, cv2.IMREAD_GRAYSCALE)

    t_l = transform(image=img_l_raw)["image"].unsqueeze(0).to(DEVICE)
    t_r = transform(image=img_r_raw)["image"].unsqueeze(0).to(DEVICE)

    # C. Generate Heatmap
    with torch.no_grad():
        heatmap_tensor = model.get_heatmap(t_l, t_r)

    # Convert to Numpy (0 to 1 range)
    heatmap = heatmap_tensor.squeeze().cpu().numpy()
    heatmap = (heatmap - heatmap.min()) / (heatmap.max() - heatmap.min() + 1e-8) # Normalize 0-1

    # D. Geometric Correction (Flip logic)
    # The heatmap is generated in "Left Foot Geometry".
    # If we want to visualize the RIGHT foot, we must flip the heatmap back.
    if TARGET_SIDE == "R":
        heatmap = cv2.flip(heatmap, 1) # Horizontal Flip
        print("🔄 Flipped heatmap to match Right Foot geometry.")
    
    # E. Load RGB Image (The Target)
    rgb_filename = f"{TARGET_ID}_{TARGET_SIDE}.png"
    rgb_path = os.path.join(RGB_DIR, rgb_filename)
    
    if not os.path.exists(rgb_path):
        print(f"❌ RGB file not found: {rgb_path}")
        return
    
    img_rgb = cv2.imread(rgb_path)
    img_rgb = cv2.cvtColor(img_rgb, cv2.COLOR_BGR2RGB) # Fix colors
    
    # Resize RGB to 256x256 to match Heatmap
    img_rgb = cv2.resize(img_rgb, (256, 256))

    # F. Overlay
    # 1. Colorize Heatmap (Blue=Safe, Red=Risk)
    heatmap_uint8 = (heatmap * 255).astype(np.uint8)
    heatmap_colored = cv2.applyColorMap(heatmap_uint8, cv2.COLORMAP_JET)
    heatmap_colored = cv2.cvtColor(heatmap_colored, cv2.COLOR_BGR2RGB)

    # 2. Blend (0.6 RGB + 0.4 Heatmap)
    overlay = cv2.addWeighted(img_rgb, 0.6, heatmap_colored, 0.4, 0)

    # G. Plotting
    plt.figure(figsize=(12, 5))

    plt.subplot(1, 3, 1)
    plt.imshow(img_rgb)
    plt.title(f"Original RGB ({TARGET_SIDE})")
    plt.axis("off")

    plt.subplot(1, 3, 2)
    plt.imshow(heatmap, cmap='jet')
    plt.title("Model Attention (Asymmetry)")
    plt.axis("off")

    plt.subplot(1, 3, 3)
    plt.imshow(overlay)
    plt.title("Fused Risk Map")
    plt.axis("off")

    save_file = f"Result_{TARGET_ID}_{TARGET_SIDE}.png"
    plt.savefig(save_file)
    print(f"🎉 Saved visualization to: {save_file}")
    plt.show()

if __name__ == "__main__":
    visualize_ulcer_zones()